In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Set environment variables to ensure CUDA is utilized
%env FORCE_CMAKE=1
%env CMAKE_ARGS=-DGGML_CUDA=ON

# Install the wheel and the server extra
!pip install llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import os

# 1. Configuration
MODEL_FILE = "Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"
TMP_PATH = os.path.join("/tmp", MODEL_FILE)
URL = f"https://huggingface.co/HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive/resolve/main/{MODEL_FILE}?download=true"

# 2. Install llama-cpp-python (Pip handles system paths, but model goes to /tmp)
!pip install llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 -q

# 3. Clean and Download to /tmp
if os.path.exists(TMP_PATH):
    os.remove(TMP_PATH)

print(f"Downloading model to {TMP_PATH}...")
!wget -c "{URL}" -O "{TMP_PATH}"

# 4. Verify Final Size
if os.path.exists(TMP_PATH):
    size_gb = os.path.getsize(TMP_PATH) / (1024**3)
    print(f"\nSuccess! File verified in /tmp. Size: {size_gb:.2f} GB")
else:
    print("\n!! Download failed to /tmp !! Check sidebar Internet settings.")

In [ ]:
from llama_cpp import Llama

# GPU T4 x2 (30GB VRAM) setup
llm = Llama(
    model_path=f"/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf",
    n_gpu_layers=-1,
    tensor_split=[0.5, 0.5],
    n_ctx=2048,
    use_mmap=False,
    flash_attn=True,
    verbose=True
)

print("\n--- Model Loaded from /tmp ---")

In [ ]:
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Hello, how are you?"}
    ],
    max_tokens=512,
    temperature=0.9
)

print("\n" + "="*30)
print(response["choices"][0]["message"]["content"])

In [ ]:
import os
from IPython.display import display, Markdown, clear_output

def chat():
    chat_history = [
        {"role": "system", "content": "You are a helpful AI assistant."}
    ]
    
    print("--- Chat Started (Type 'exit' to stop) ---")
    
    while True:
        user_input = input("User: ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        chat_history.append({"role": "user", "content": user_input})
        
        response = llm.create_chat_completion(
            messages=chat_history,
            max_tokens=2048,
            temperature=0.8,
            stop=["<|im_end|>", "<|endoftext|>"]
        )
        
        answer = response["choices"][0]["message"]["content"]
        chat_history.append({"role": "assistant", "content": answer})
        
        display(Markdown(f"**Assistant:**\n\n{answer}"))
        print("-" * 50)

chat()

In [ ]:
!pip install pyngrok -q

In [ ]:
import os
import signal
import subprocess

try:
    p = subprocess.Popen(['fuser', '-v', '/dev/nvidia0', '/dev/nvidia1'], stdout=subprocess.PIPE)
    out, err = p.communicate()
    for line in out.split():
        os.kill(int(line), signal.SIGKILL)
    print("VRAM Purged.")
except:
    print("No ghost processes found. Ready to go.")

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok

# --- CONFIGURATION ---
NGROK_TOKEN = "Your_Ngrok_Token_Here" # <--- Paste your ngrok token here
MODEL_PATH = "/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

ngrok.set_auth_token(NGROK_TOKEN)

server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", MODEL_PATH,
    "--n_gpu_layers", "-1",
    "--tensor_split", "0.5", "0.5",
    "--n_ctx", "2048",
    "--use_mmap", "False",
    "--host", "0.0.0.0",
    "--port", "8000"
]

print("Starting API Server... please wait.")
server_process = subprocess.Popen(server_cmd)

time.sleep(30) 
public_url = ngrok.connect(8000).public_url

print(f"\n" + "="*50)
print(f"API IS LIVE AT: {public_url}")
print("="*50)

In [ ]:
from IPython.display import display, Markdown
import openai

client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="sk-no-key-required")

def chat_interface():
    system_msg = {"role": "system", "content": "You are a helpful AI assistant."}
    history = []
    
    print("--- Chat (Type 'exit' to quit) ---")
    
    while True:
        user_msg = input("\nUser: ")
        if user_msg.lower() in ['exit', 'quit']: break
        
        history.append({"role": "user", "content": user_msg})
        messages_to_send = [system_msg] + history[-6:] 
        
        try:
            completion = client.chat.completions.create(
                model="qwen3.5",
                messages=messages_to_send,
                max_tokens=800, 
                temperature=0.8
            )
            
            answer = completion.choices[0].message.content
            history.append({"role": "assistant", "content": answer})
            display(Markdown(f"### Assistant:\n---\n{answer}"))
            
        except Exception as e:
            if "context_length_exceeded" in str(e):
                print("!! Error: Conversation too long. Auto-trimming history...")
                history = history[-2:] 
            else:
                print(f"Error: {e}")

chat_interface()